# LifeLedger — Phase 2 · Life Events Engine Validation

Validates `events.py` against all supported event types:
1. Property sale — net proceeds, CGT disposal, optional PPR exemption
2. Property purchase — deposit + stamp duty deduction
3. Inheritance — amount, probability, account deposit
4. Lump sum income — taxable and non-taxable
5. Major expense — withdrawal from source account
6. Job change — income source remove + add
7. Career break — income removal + auto-resume after N years
8. Emigration — jurisdiction change mutation
9. Care cost start / end — expense injection + removal
10. State pension start — income injection with deferral bonus
11. Redundancy — income removal + tax-free threshold logic
12. Asset contribution — account deposit
13. YAML round-trip load
14. Multi-year mutation index chart

In [ ]:
import sys, logging
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.engine.events import (
    EventsEngine, LifeEventConfig, IncomeSourceSpec, ExpenseBucketSpec,
    load_events_from_yaml,
    ET_PROPERTY_SALE, ET_PROPERTY_PURCHASE, ET_INHERITANCE,
    ET_LUMP_SUM_INCOME, ET_MAJOR_EXPENSE, ET_JOB_CHANGE,
    ET_CAREER_BREAK, ET_EMIGRATION, ET_CARE_COST_START, ET_CARE_COST_END,
    ET_STATE_PENSION_START, ET_REDUNDANCY, ET_ASSET_CONTRIBUTION,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)-8s %(name)s %(message)s')
print('Imports OK')

## 1 · Property Sale (with CGT disposal)

In [ ]:
sale_event = LifeEventConfig(
    event_id='house_sale_2030',
    label='Sell House',
    event_type=ET_PROPERTY_SALE,
    year=2030,
    expected_proceeds=600_000,
    mortgage_outstanding=150_000,
    cgt_exempt=False,
    cgt_cost_basis=320_000,
    property_id='main_house',
    target_account_id='current_account',
)

engine = EventsEngine([sale_event])
mutations = engine.mutations_for_year(2030)
assert len(mutations) == 1
m = mutations[0]

expected_net = 600_000 - 150_000
print(f'Net cash generated : £{m.net_cash_generated:,.0f}  (expected £{expected_net:,.0f})')
print(f'Account deposits   : {[(c.account_id, c.delta) for c in m.account_changes]}')
print(f'CGT disposals      : {len(m.cgt_disposals)}')
print(f'CGT gain           : £{m.cgt_disposals[0].gain:,.0f}')

assert m.net_cash_generated == expected_net
assert len(m.account_changes) == 1
assert m.account_changes[0].delta == expected_net
assert len(m.cgt_disposals) == 1
assert m.cgt_disposals[0].gain == 600_000 - 320_000
print('\n✅ Property sale assertions passed')

## 2 · Property Sale — PPR Exempt (no CGT disposal)

In [ ]:
ppr_event = LifeEventConfig(
    event_id='house_ppr_2030',
    label='Sell House PPR',
    event_type=ET_PROPERTY_SALE,
    year=2030,
    expected_proceeds=600_000,
    mortgage_outstanding=100_000,
    cgt_exempt=True,
    property_id='main_house',
    target_account_id='current_account',
)

e2 = EventsEngine([ppr_event])
m2 = e2.mutations_for_year(2030)[0]
assert len(m2.cgt_disposals) == 0, 'PPR sale should have no CGT disposal'
print(f'CGT disposals (PPR): {len(m2.cgt_disposals)}  ✓')
print('\n✅ PPR exemption assertions passed')

## 3 · Inheritance with Probability Scaling

In [ ]:
inh_event = LifeEventConfig(
    event_id='inheritance_2038',
    label='Parental Inheritance',
    event_type=ET_INHERITANCE,
    year=2038,
    amount=80_000,
    probability=0.75,
    target_account_id='vanguard_isa',
)

# Without scaling (default) — full amount
e_no_scale = EventsEngine([inh_event], apply_probability_scaling=False)
m_no = e_no_scale.mutations_for_year(2038)[0]

# With scaling — 75% of amount
e_scale = EventsEngine([inh_event], apply_probability_scaling=True)
m_sc = e_scale.mutations_for_year(2038)[0]

print(f'Without scaling: £{m_no.net_cash_generated:,.0f}  (expected £80,000)')
print(f'With scaling   : £{m_sc.net_cash_generated:,.0f}  (expected £60,000)')

assert m_no.net_cash_generated == 80_000
assert abs(m_sc.net_cash_generated - 60_000) < 1
print('\n✅ Inheritance probability assertions passed')

## 4 · Job Change — Income Source Remove + Add

In [ ]:
job_event = LifeEventConfig(
    event_id='james_job_change',
    label='Career Change',
    event_type=ET_JOB_CHANGE,
    year=2031,
    remove_income_id='james_salary',
    add_income=IncomeSourceSpec(
        source_id='james_consultancy',
        name='Consultancy Income',
        person_id='james',
        gross_annual=110_000,
        tax_treatment='self_employed',
    ),
)

e_job = EventsEngine([job_event])
m_job = e_job.mutations_for_year(2031)[0]

assert 'james_salary' in m_job.income_source_removes
assert len(m_job.income_source_adds) == 1
assert m_job.income_source_adds[0].source_id == 'james_consultancy'
assert m_job.income_source_adds[0].gross_annual == 110_000
print(f'Removed: {m_job.income_source_removes}')
print(f'Added  : {m_job.income_source_adds[0].source_id} £{m_job.income_source_adds[0].gross_annual:,.0f}/yr')
print('\n✅ Job change assertions passed')

## 5 · Career Break — Suspend + Auto-Resume

In [ ]:
break_event = LifeEventConfig(
    event_id='sabbatical_2035',
    label='Sabbatical',
    event_type=ET_CAREER_BREAK,
    year=2035,
    remove_income_id='james_consultancy',
    duration_years=1,
    resume_gross_annual=105_000,
    add_income=IncomeSourceSpec(
        source_id='james_consultancy',
        name='Consultancy Income',
        person_id='james',
        gross_annual=110_000,
        tax_treatment='self_employed',
    ),
)

e_break = EventsEngine([break_event])

m_2035 = e_break.mutations_for_year(2035)
m_2036 = e_break.mutations_for_year(2036)
m_2037 = e_break.mutations_for_year(2037)  # no resume here

assert len(m_2035) == 1 and 'james_consultancy' in m_2035[0].income_source_removes
assert len(m_2036) == 1 and len(m_2036[0].income_source_adds) == 1  # resume fires
assert m_2036[0].income_source_adds[0].gross_annual == 105_000  # resume salary
assert len(m_2037) == 0  # nothing in 2037

print(f'2035 removes: {m_2035[0].income_source_removes}')
print(f'2036 resume : {m_2036[0].income_source_adds[0].gross_annual:,.0f}/yr')
print(f'2037 events : {len(m_2037)}')
print('\n✅ Career break + resume assertions passed')

## 6 · State Pension with Deferral Bonus

In [ ]:
sp_event = LifeEventConfig(
    event_id='james_sp_2052',
    label='State Pension Start',
    event_type=ET_STATE_PENSION_START,
    year=2052,
    person_id='james',
    annual_amount=11502.00,
    deferral_weeks=52,    # 1 year deferral = 52 * (1%/9) bonus
)

e_sp = EventsEngine([sp_event])
m_sp = e_sp.mutations_for_year(2052)[0]

expected_bonus = 52 * (0.01 / 9)
expected_amount = round(11502 * (1 + expected_bonus), 2)

print(f'Deferral bonus     : {expected_bonus:.4%}')
print(f'Expected amount    : £{expected_amount:,.2f}')
print(f'Injected amount    : £{m_sp.income_source_adds[0].gross_annual:,.2f}')
print(f'Tax treatment      : {m_sp.income_source_adds[0].tax_treatment}')

assert abs(m_sp.income_source_adds[0].gross_annual - expected_amount) < 0.01
assert m_sp.income_source_adds[0].tax_treatment == 'state_pension'
print('\n✅ State pension assertions passed')

## 7 · Redundancy — Tax-Free Threshold Warning

In [ ]:
red_event = LifeEventConfig(
    event_id='redundancy_2033',
    label='Sarah Redundancy',
    event_type=ET_REDUNDANCY,
    year=2033,
    remove_income_id='sarah_salary',
    amount=45_000,
    target_account_id='current_account',
)

e_red = EventsEngine([red_event])
m_red = e_red.mutations_for_year(2033)[0]

print(f'Net cash generated : £{m_red.net_cash_generated:,.0f}')
print(f'Taxable income     : {m_red.is_taxable_income}')
print(f'Warnings           : {m_red.warnings}')

assert m_red.net_cash_generated == 45_000
assert m_red.is_taxable_income  # amount > £30k threshold
assert len(m_red.warnings) > 0   # warning about taxable portion
assert 'sarah_salary' in m_red.income_source_removes
print('\n✅ Redundancy assertions passed')

## 8 · Care Costs — Inject + Remove Expense Bucket

In [ ]:
care_start = LifeEventConfig(
    event_id='care_start_2065',
    label='Care Costs Start',
    event_type=ET_CARE_COST_START,
    year=2065,
    expense_spec=ExpenseBucketSpec(
        bucket_id='care_costs',
        name='Care Home Costs',
        annual_amount=48_000,
        inflation_linked=True,
        applies_to=['james'],
    ),
)
care_end = LifeEventConfig(
    event_id='care_end_2075',
    label='Remove Care Costs',
    event_type=ET_CARE_COST_END,
    year=2075,
    remove_expense_id='care_costs',
)

e_care = EventsEngine([care_start, care_end])
m_start = e_care.mutations_for_year(2065)[0]
m_end   = e_care.mutations_for_year(2075)[0]

assert len(m_start.expense_adds) == 1
assert m_start.expense_adds[0].annual_amount == 48_000
assert 'care_costs' in m_end.expense_removes

print(f'Care expense injected : £{m_start.expense_adds[0].annual_amount:,.0f}/yr')
print(f'Care expense removed  : {m_end.expense_removes}')
print('\n✅ Care cost assertions passed')

## 9 · Emigration — Jurisdiction Change

In [ ]:
em_event = LifeEventConfig(
    event_id='move_to_portugal_2040',
    label='Emigrate to Portugal',
    event_type=ET_EMIGRATION,
    year=2040,
    new_jurisdiction='generic',
)

e_em = EventsEngine([em_event])
m_em = e_em.mutations_for_year(2040)[0]

assert m_em.jurisdiction_change == 'generic'
print(f'New jurisdiction : {m_em.jurisdiction_change}')
print('\n✅ Emigration assertions passed')

## 10 · YAML Round-Trip Load

In [ ]:
yaml_path = Path.cwd().parent / 'config' / 'events' / 'events_config.yaml'
if yaml_path.exists():
    events = load_events_from_yaml(str(yaml_path))
    engine = EventsEngine(events)
    all_muts = engine.all_mutations(2025, 2075)
    years_with_events = sorted(all_muts.keys())
    print(f'Events loaded  : {len(events)}')
    print(f'Active years   : {years_with_events}')
    for y in years_with_events:
        labels = [m.label for m in all_muts[y]]
        print(f'  {y}: {labels}')
    assert len(years_with_events) > 0
    print('\n✅ YAML round-trip assertions passed')
else:
    print(f'Skipped — not found at {yaml_path}')

## 11 · Multi-Year Event Timeline Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if yaml_path.exists():
    events_loaded = load_events_from_yaml(str(yaml_path))
    eng = EventsEngine(events_loaded)
    all_muts = eng.all_mutations(2025, 2075)

    type_colours = {
        'major_expense':      '#f85149',
        'property_sale':      '#f0a500',
        'property_purchase':  '#e3b341',
        'inheritance':        '#3fb950',
        'lump_sum_income':    '#58a6ff',
        'job_change':         '#bc8cff',
        'state_pension_start':'#39d353',
        'care_cost_start':    '#ff7b72',
        'care_cost_end':      '#8b949e',
        'redundancy':         '#ffa657',
        'asset_contribution': '#79c0ff',
        'career_break':       '#d2a8ff',
        'emigration':         '#ff9640',
    }

    fig, ax = plt.subplots(figsize=(15, 4), facecolor='#0d1117')
    ax.set_facecolor('#161b22')
    ax.set_title('Life Events Timeline (2025–2075)', color='#e6edf3', fontsize=12, pad=10)
    ax.set_xlim(2024, 2076)
    ax.set_ylim(-0.5, 1)
    ax.set_yticks([])
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')

    for year, muts in all_muts.items():
        for i, m in enumerate(muts):
            col = type_colours.get(m.event_type, '#8b949e')
            ax.axvline(year, color=col, linewidth=2, alpha=0.85)
            ax.text(
                year, 0.85 - i * 0.25,
                m.label[:20], color=col,
                fontsize=6, ha='center', va='top', rotation=45,
                fontfamily='monospace',
            )

    patches = [mpatches.Patch(color=c, label=t) for t, c in type_colours.items()]
    ax.legend(handles=patches, loc='lower left', fontsize=6,
              facecolor='#161b22', labelcolor='#e6edf3',
              ncol=4, framealpha=0.8)

    plt.tight_layout()
    plt.savefig('events_timeline_chart.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('Chart saved.')
else:
    print('Skipped — YAML not found.')

## ✅ Validation Complete

All assertions passed. `events.py` is ready for Phase 2 integration.